# 04 Panahon, lugar, tao — notebook

Malinis na ang line list (kabanata 03). Ngayon, inilalarawan ni Maya ang outbreak sa
paraang sinisimulan ng bawat field epidemiologist: **kailan** nagkasakit ang mga tao,
**saan** sila nakatira, at **sino** sila? Bawat numerong makikita mo rito ay kinuwenta
mula sa synthetic na datos ng Santa Luz — imbento para sa pagtuturo ang Santa Luz City,
ang Bagyong Amihan at ang lahat ng kaso.

Gagawa ka ng: table ng case classification, epidemic curve na may marka ng bagyo, attack
rate bawat 10,000 ayon sa band at barangay, age–sex chart at case fatality rate.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## Hakbang 1 · I-load ang malinis na line list at ang mga denominator

Dalawang file: ang **mga kaso** (`santa_luz_linelist.csv`, isang row bawat kaso) at ang
**populasyon** ng bawat barangay (`santa_luz_barangays.csv`). Sinasabi natin sa pandas
kung aling mga column ang petsa para basahin ang mga ito bilang totoong petsa, hindi text.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

DATE_COLUMNS = ["date_onset", "date_consult", "date_report", "date_lab_confirmed"]
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv", parse_dates=DATE_COLUMNS)
barangays = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")

LANDFALL = pd.Timestamp("2026-07-18")  # Typhoon Amihan
BANDS = ["riverside", "coastal", "upland"]

print(f"{len(ll)} kaso | {len(barangays)} barangay | {barangays['population'].sum():,} residente")
ll["disease"].value_counts()

## Hakbang 2 · Case classification

Na-classify ang bawat row ayon sa case definition sa data freeze noong 30 Setyembre:
**suspect** (pasok sa clinical description), **probable** (dagdag ang positibong rapid
test), **confirmed** (dagdag ang positibong laboratory result). Binibilang ng cross-table
ang bawat kombinasyon; nagdadagdag ng total ang `margins=True`.

In [ ]:
classes = pd.crosstab(ll["disease"], ll["case_class"], margins=True, margins_name="Total")
classes[["confirmed", "probable", "suspect", "Total"]]

## Hakbang 3 · Ang epidemic curve (panahon)

> **Tanungin si Tala:** "May datetime column na `date_onset` ang DataFrame kong `lepto`
> (gawa-gawang rows: `2026-07-24`, `2026-07-24`, `2026-07-27`). Gumuhit ng epidemic curve:
> isang bar bawat araw mula 15 Hulyo hanggang 31 Agosto 2026, **kasama ang mga araw na
> zero ang kaso**, walang puwang sa pagitan ng mga bar. Markahan ang 18 Hulyo ng pulang
> putol-putol na linya na may label na 'Landfall ng Bagyong Amihan', lagyan ng shading ang
> mga araw ng baha 18–20 Hulyo at ang clean-up 21 Hulyo – 2 Agosto."

Ang mahalagang diskarte ay ang `reindex(..., fill_value=0)`: inililista lang ng
`value_counts()` ang mga araw na *may* kaso, kaya inilalatag natin ang mga bilang na iyon
sa buong kalendaryo.

In [ ]:
lepto = ll[ll["disease"] == "leptospirosis"]
days = pd.date_range("2026-07-15", "2026-08-31", freq="D")
daily = lepto["date_onset"].value_counts().reindex(days, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axvspan(pd.Timestamp("2026-07-17 12:00"), pd.Timestamp("2026-07-20 12:00"),
           color="#4FA3D9", alpha=0.30, label="Baha (18–20 Hul)")
ax.axvspan(pd.Timestamp("2026-07-20 12:00"), pd.Timestamp("2026-08-02 12:00"),
           color="#FCD116", alpha=0.25, label="Clean-up (21 Hul – ~2 Ago)")
ax.bar(daily.index, daily.values, width=1.0, color="#0038A8", edgecolor="white", linewidth=0.6)
ax.axvline(LANDFALL, color="#CE1126", linestyle="--", linewidth=2,
           label="Landfall ng Bagyong Amihan (18 Hul)")
ax.set_title("Kaso ng leptospirosis ayon sa petsa ng onset, Santa Luz City (synthetic na datos)")
ax.set_xlabel("Petsa ng onset (2026)")
ax.set_ylabel("Bilang ng kaso")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

**I-check natin.** Dapat tumugma ang kabuuan ng mga bar sa bawat kaso ng leptospirosis na
**may** petsa ng onset (2 ang wala — kabanata 03, rule C4). Dapat tumugma ang lingguhang
total sa outbreak report. At hanapin ang peak day.

In [ ]:
missing_onset = lepto["date_onset"].isna().sum()
assert daily.sum() == len(lepto) - missing_onset
print(f"{daily.sum()} sa mga bar + {missing_onset} na walang onset = {len(lepto)} kaso")

weekly = lepto["date_onset"].dt.isocalendar().week.value_counts().sort_index()
weekly.index = weekly.index.astype(int)
print("Kaso ayon sa ISO week ng onset:", weekly.to_dict())
assert weekly.to_dict() == {30: 35, 31: 108, 32: 68, 33: 15, 34: 2}
print("Peak day:", daily.idxmax().date(), "na may", daily.max(), "kaso")

## Hakbang 4 · Ano ang sinasabi ng hugis?

Basahin ang curve na parang detective:

- **22 Hulyo ang unang onset**, apat na araw pagkatapos ng landfall (nangongolekta ang
  enhanced line list ng onset mula 20 Hulyo). Kinailangan ng panahon bago magkasakit ang
  mga tao: ang **incubation period**, ang panahon mula impeksiyon hanggang sa unang
  sintomas.
- **Iisang alon**: mabilis na tumaas, nanatiling mataas nang mga dalawang linggo (24 Hulyo
  – 6 Agosto), saka humupa pagdating ng kalagitnaan ng Agosto. Walang ikalawa, ikatlo o
  ikaapat na alon.
- Karaniwang hindi kumakalat ang leptospirosis mula tao sa tao, at kasabay ng alon ang
  baha at ang clean-up, noong patuloy na lumulusong ang mga tao sa tubig na posibleng
  kontaminado.

Iyan ang palatandaan ng **common source** na na-expose ang mga tao sa loob ng ilang araw
— isang **continuing common source** — hindi iisang sandali (mas matulis pang iisang peak
ang ibibigay ng *point source*) at hindi *propagated* na outbreak (sunod-sunod na alon,
isang incubation period ang pagitan ng bawat isa). Nagiging konkreto ang pangangatwiran
kapag binilang ang mga araw mula sa landfall.

Para may maikumpara, iginuguhit din natin ang curve ng **dengue** ayon sa linggo: dala ng
lamok na *Aedes* ang dengue, na nangingitlog sa naiipong tubig, kaya mas huli itong
tumataas at ilang linggo pang patuloy na tumataas.

In [ ]:
days_after = (lepto["date_onset"] - LANDFALL).dt.days.dropna().astype(int)
print("Araw mula landfall hanggang onset: una", days_after.min(), "| median", int(days_after.median()),
      "| huli", days_after.max())

dengue = ll[ll["disease"] == "dengue"]
dengue_weeks = dengue["date_onset"].dt.isocalendar().week.value_counts().reindex(range(30, 40), fill_value=0)
print("Kaso ng dengue ayon sa ISO week ng onset:", dengue_weeks.to_dict())

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([f"W{w}" for w in dengue_weeks.index], dengue_weeks.values, width=0.95, color="#00737D")
ax.set_title("Kaso ng dengue ayon sa linggo ng onset, 2026 (synthetic; kulang pa ang W39)")
ax.set_xlabel("ISO week ng onset")
ax.set_ylabel("Bilang ng kaso")
plt.tight_layout()
plt.show()

## Hakbang 5 · Nagkamali si Tala: "attack rate" na walang populasyon

> **Tanungin si Tala:** "I-compute ang dengue attack rate ayon sa `band`."

Tumatakbo at mukhang maayos ang sagot ni Tala:

In [ ]:
# Unang sagot ni Tala: bahagi ng mga KASO sa bawat band -- HINDI attack rate
wrong = (dengue["band"].value_counts(normalize=True) * 100).round(1)
print("'Attack rate (%)' ni Tala:")
print(wrong)
riverside_share = (lepto["band"] == "riverside").mean() * 100
print(f"Parehong mali sa leptospirosis: {riverside_share:.0f}% ng kaso ay taga-tabing-ilog")

100 % ang kabuuan ng mga numerong ito. Iyan ang palatandaan: sinasabi nito **kung nasaan
ang mga kaso**, hindi **gaano kadelikadong tumira roon**. Pinakamarami ang kaso ng dengue
sa tabing-ilog dahil lang ito ang may pinakamaraming tao (96,064 sa 186,000). Ang
**attack rate** ay

$$\text{attack rate} = \frac{\text{bagong kaso sa isang grupo}}{\text{mga taong at risk sa grupong iyon}}$$

at para sa isang lungsod, ipinapahayag natin ito **bawat 10,000 residente**. Galing sa
barangay file ang denominator, hindi sa line list.

In [ ]:
population = barangays.groupby("band")["population"].sum()
cases = pd.crosstab(ll["band"], ll["disease"])

attack = cases.div(population, axis=0) * 10_000
attack.loc["city"] = cases.sum() / population.sum() * 10_000
attack = attack.loc[BANDS + ["city"], ["leptospirosis", "dengue"]].round(1)
print("Attack rate bawat 10,000 residente")
attack

**I-check natin** nang mano-mano ang isang cell, at ikumpara ang dalawang sagot. Para sa dengue,
nauna ang tabing-ilog (riverside) sa bersiyon ni Tala; sinasabi ng totoong attack rate na
ang **baybayin** (coastal) ang may pinakamataas na panganib.

In [ ]:
riverside_lepto = (ll["disease"].eq("leptospirosis") & ll["band"].eq("riverside")).sum()
print(f"Leptospirosis sa tabing-ilog: {riverside_lepto} / {population['riverside']:,} x 10,000 =",
      round(riverside_lepto / population["riverside"] * 10_000, 1))
assert attack.loc["riverside", "leptospirosis"] == 19.7
assert attack.loc["coastal", "dengue"] == 29.6
assert attack["dengue"].drop("city").idxmax() == "coastal" != wrong.idxmax()

## Hakbang 6 · Lugar: attack rate ayon sa barangay

Ngayon, parehong ideya para sa bawat isa sa 24 na barangay. **Nagsisimula tayo sa
barangay table** at idinudugtong dito ang bilang ng kaso, para lumabas pa rin ang
barangay na walang kaso (ang Bukid sa kabundukan) — na may rate na 0, hindi missing.

In [ ]:
counts = ll.groupby(["barangay_id", "disease"]).size().unstack(fill_value=0)
place = barangays.set_index("barangay_id")[["barangay", "band", "population"]].join(counts).fillna(0)
place[["leptospirosis", "dengue"]] = place[["leptospirosis", "dengue"]].astype(int)
for disease in ["leptospirosis", "dengue"]:
    place[f"ar_{disease}"] = (place[disease] / place["population"] * 10_000).round(1)

print(len(place), "barangay;", (place["leptospirosis"] + place["dengue"] == 0).sum(), "ang walang kaso")
top = place.sort_values("ar_leptospirosis", ascending=False)
top[["barangay", "band", "leptospirosis", "population", "ar_leptospirosis"]].head(4)

Ipinapakita sa isang tingin ng bar chart na nakaayos ayon sa rate at may kulay ayon sa
band ang pattern (iginuguhit ito bilang mapa ng kabanata 08). Sa legend: riverside =
tabing-ilog, coastal = baybayin, upland = kabundukan.

In [ ]:
BAND_COLOURS = {"riverside": "#0038A8", "coastal": "#00737D", "upland": "#2F7D32"}
fig, axes = plt.subplots(1, 2, figsize=(11, 6.5), sharey=False)
for ax, disease in zip(axes, ["leptospirosis", "dengue"]):
    data = place.sort_values(f"ar_{disease}")
    ax.barh(data["barangay"], data[f"ar_{disease}"], color=data["band"].map(BAND_COLOURS))
    ax.set_title(disease.capitalize())
    ax.set_xlabel("Kaso bawat 10,000 residente")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in BAND_COLOURS.values()]
fig.legend(handles, BAND_COLOURS.keys(), loc="lower center", ncol=3)
fig.suptitle("Attack rate ayon sa barangay, Santa Luz City (synthetic na datos)")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

print("Dengue, dalawang pinakamataas:", place.nlargest(2, "ar_dengue")[["barangay", "ar_dengue"]].values.tolist())

## Hakbang 7 · Tao: edad, sex at mga namatay

Sino ang nagkasakit? Isang magkatabing **age–sex chart** (lalaki sa kaliwa, babae sa
kanan) para sa bawat sakit, kasama ang median na edad at ang bahagi ng mga lalaki. Pitong
kaso ng dengue ang walang edad, kaya hiwalay natin silang binibilang imbes na hayaang
mawala.

Saka ang **case fatality rate** (CFR): sa mga taong naging kaso, ilang bahagi ang
namatay? Dito, ang mga kaso *mismo* ang denominator — iyan ang depinisyon, at kabaligtaran
ng pagkakamali sa attack rate.

In [ ]:
AGE_GROUPS = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, disease in zip(axes, ["leptospirosis", "dengue"]):
    d = ll[ll["disease"] == disease]
    table = pd.crosstab(d["age_group"], d["sex"]).reindex(AGE_GROUPS, fill_value=0)
    ax.barh(AGE_GROUPS, -table["M"], color="#0038A8", label="Lalaki")
    ax.barh(AGE_GROUPS, table["F"], color="#CE1126", label="Babae")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{disease.capitalize()} (n = {len(d)})")
    ax.set_xlabel("Kaso (lalaki sa kaliwa, babae sa kanan)")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{abs(int(x))}")
axes[0].set_ylabel("Age group (taon)")
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

for disease in ["leptospirosis", "dengue"]:
    d = ll[ll["disease"] == disease]
    male = (d["sex"] == "M").sum()
    print(f"{disease:<14} median na edad {d['age'].median():.0f} | lalaki {male} ({male / len(d):.0%})"
          f" | walang edad {d['age'].isna().sum()}")

# Case fatality rate: namatay / KASO (dito, ang mga kaso MISMO ang tamang denominator)
cfr = ll.groupby("disease").agg(cases=("case_id", "size"),
                                deaths=("outcome", lambda s: (s == "died").sum()),
                                hospitalised=("hospitalised", "sum"))
cfr["cfr_pct"] = (cfr["deaths"] / cfr["cases"] * 100).round(1)
cfr["hospitalised_pct"] = (cfr["hospitalised"] / cfr["cases"] * 100).round(1)
assert cfr.loc["leptospirosis", "deaths"] == 20 and cfr.loc["leptospirosis", "cfr_pct"] == 8.7
cfr

## Ikaw naman

1. **Dengue epi curve ayon sa araw.** Kopyahin ang cell ng leptospirosis curve at iguhit
   ang dengue ayon sa araw ng onset mula 15 Hulyo hanggang 30 Setyembre. Mukha ba itong
   iisang alon mula sa iisang pinagmulan? *Hint: palitan ang filter at ang `date_range`;
   alisin ang shading ng clean-up.*
2. **Attack rate ayon sa sex.** Walang hati ayon sa sex ang barangay file, kaya ipagpalagay
   na kalahati ng populasyon ay lalaki. I-compute ang kaso ng leptospirosis bawat 10,000
   lalaki at bawat 10,000 babae. *Hint: `lepto["sex"].value_counts() / (186_000 / 2) * 10_000`,
   at sabihin nang malinaw sa tala mo na assumption ang denominator.*
3. **CFR ayon sa age group.** Mas nakamamatay ba ang leptospirosis sa mas matatandang kaso?
   *Hint: `lepto.groupby("age_group")["outcome"].apply(lambda s: (s == "died").mean())` — at
   i-print din ang bilang ng kaso sa bawat grupo, dahil alanganin ang CFR mula sa 9 na kaso.*